In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
# get_peft_model LoraConfig LoRA配置类  TaskType
from peft import get_peft_model, LoraConfig, TaskType,prepare_model_for_kbit_training,PeftConfig,PeftModel
# trainder 简化训练代码 TrainingArguments 设置训练过程超参数
from transformers import Trainer, TrainingArguments
# 处理语言模型训练中的数据批次
from transformers import DataCollatorForLanguageModeling
from datasets import load_dataset
import json
import torch


c:\Users\24167\.conda\envs\lora\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0921 12:54:43.820000 43704 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [ ]:
# 预训练模型名字
model_name ="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# 加载对应分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 加载预训练模型  device_map "auto"表示自动分配设备 dtype=torch.float16表示使用半精度浮点数
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", dtype=torch.float16)

model

In [ ]:
# 加载数据集（先取 200 条跑通流程；正式训练可去掉 [:200] 或加大到几万条）
dataset = load_dataset("MBZUAI/LaMini-instruction", split="train[:200]")
# 检查数据格式
print(dataset.column_names)
print(dataset[:5])

In [ ]:
#数据预处理

# 手动分割数据集为训练集和验证集 90% 训练集 10% 验证集
train_dataset = dataset.train_test_split(test_size=0.1, seed=42)
# 训练集
train_dataset_raw = train_dataset["train"]
# 验证集
eval_dataset_raw = train_dataset["test"]

# 定义数据集处理函数（修正版：拼接指令+回应，并把指令部分和 padding 的标签置 -100）
def preprocess_funcation(examples):
    # 1) 指令 + 回应 拼成完整一段文本
    texts = [f"指令:{ins}\n回应:{res}" for ins, res in zip(examples["instruction"], examples["response"])]

    # 2) 一次性编码完整文本
    tokenized = tokenizer(texts, max_length=512, truncation=True, padding="max_length")

    # 3) 标签 = input_ids 副本，指令部分和 padding 置 -100（不参与 loss）
    labels = []
    for i, ins in enumerate(examples["instruction"]):
        prompt_len = len(tokenizer(f"指令:{ins}\n回应:")["input_ids"])  # 指令部分 token 数
        row = tokenized["input_ids"][i][:]
        for j in range(len(row)):
            if j < prompt_len or tokenized["attention_mask"][i][j] == 0:
                row[j] = -100
        labels.append(row)
    tokenized["labels"] = labels
    return tokenized


# 处理训练集和验证集
train_dataset = train_dataset_raw.map(preprocess_funcation, batched=True, remove_columns=train_dataset_raw.column_names)
eval_dataset = eval_dataset_raw.map(preprocess_funcation, batched=True, remove_columns=eval_dataset_raw.column_names)

train_dataset
eval_dataset

In [ ]:
#设置LoRA参数

# LoRA配置
lora_config = LoraConfig(
    r=8,                                              # 低秩矩阵的秩
    lora_alpha=32,                                    # LoRA 的缩放因子
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # 目标模块：注意力投影层
    lora_dropout=0.1,                                 # LoRA 的 dropout 率
    bias="none",                                      # 偏置类型
    task_type=TaskType.CAUSAL_LM,                     # 任务类型：因果语言模型
)

# 将 LoRA 配置应用到模型上（fp16 全精度训练，无需 prepare_model_for_kbit_training）
peft_model = get_peft_model(model, lora_config)

peft_model.print_trainable_parameters()

In [ ]:
#模型训练

training_args = TrainingArguments(
    output_dir="./results",              # 输出目录
    num_train_epochs=3,                  # 训练轮数
    learning_rate=2e-4,                  # 学习率
    per_device_train_batch_size=2,       # 训练批次大小（8GB 显存用 2 更稳）
    per_device_eval_batch_size=2,        # 验证批次大小
    gradient_accumulation_steps=4,       # 梯度累积步数（等效 batch=8）
    weight_decay=0.01,                   # 权重衰减
    fp16=True,                           # 混合精度训练，省显存提速
    eval_strategy="epoch",               # 验证策略（新版 transformers 用 eval_strategy）
    save_strategy="epoch",               # 保存策略
    load_best_model_at_end=True,         # 训练结束后加载最佳模型
    save_total_limit=2,                  # 只保留最近 2 个 checkpoint
    push_to_hub=False,                   # 是否推送到 Hugging Face Hub
    report_to="none",                    # 关闭 tensorboard/wandb（未安装时避免报错）
    logging_steps=10,                    # 每 10 步打印一次 loss
    label_names=["labels"],              # 标签列名
)

# 初始化 Trainer（transformers 5.x 里 tokenizer 参数改名 processing_class）
trainer = Trainer(
    model=peft_model,                    # 模型
    args=training_args,                  # 训练参数
    train_dataset=train_dataset,         # 训练集
    eval_dataset=eval_dataset,           # 验证集
    processing_class=tokenizer,          # 分词器
)

# 开始训练
trainer.train()

In [9]:
# ===== 加载并合并微调后的 LoRA 模型 =====

# 1) LoRA 权重所在目录（训练脚本里 save_pretrained 保存的）
peft_model_path = "./lora_model"

# 2) 读取 LoRA 配置，从中拿到对应的基础模型名字
config = PeftConfig.from_pretrained(peft_model_path)

# 3) 先加载「基础模型」（未微调的原始权重）
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="auto",
    dtype=torch.float16,
)

# 4) 把 LoRA 权重「挂」到基础模型上，得到带 adapter 的模型
model = PeftModel.from_pretrained(model, peft_model_path)

# 5) 加载分词器
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# 6) 合并并卸载 LoRA 权重：
#    把 LoRA 学到的增量 (alpha/r)*B@A 直接加进原始权重，即 W' = W + (alpha/r)*B@A，
#    然后拆掉 LoRA 结构，得到一个普通的、完整的模型
model = model.merge_and_unload()

# 7) 保存合并后的完整模型
model.save_pretrained('./merged_model')
tokenizer.save_pretrained('./merged_model')


# ===== 测试合并后的模型 =====
def generate_text(input_text):
    # 用训练时一致的模板构造提示词
    prompt = f"指令:{input_text}\n回应:"
    # 编码并移到 GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # 关闭梯度计算，加速推理
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            pad_token_id=tokenizer.eos_token_id,
        )
    # 解码，并取出「回应:」之后的部分
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response.split("回应:")[-1].strip()

# 测试
input_text = "List 3 benefits of regular exercise"
response = generate_text(input_text)
print(response)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.39s/it]


1. Improved physical fitness
2. Reduced risk of chronic diseases
3. Enhanced mental and emotional well-being
